In [ ]:
import time
t0 = time.time()

print(time.time() - t0)

import os
print(os.getcwd())

!ls -la ./use_Hybrid_Data


1.621246337890625e-05
/Users/debajyotihazra/Documents/Modify RAG learning/HYbrid RAG
ls: ./use_Hybrid_Data: No such file or directory


In [ ]:
# Loader load 
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('Why_Language_Models_Hallucinate_Explainer.pdf')
pages = loader.load()

# Chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap = 120)
texts = text_spliter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]

# VectorDB create 
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="./use_hybrid_Data")
collection = client.get_or_create_collection(name="HYbrid_RAG",embedding_function=embedding_function)
embedding_function(["test sentence"])

if collection.count()==0:
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))],
        metadatas=metadata
    )

import string
from rank_bm25 import BM25Okapi
tokens = [c.split() for c in chunks]
bm250 = BM25Okapi(tokens)
